In [1]:
import os 
from dotenv import load_dotenv

## PART 1 — Q&A Chatbot using OpenAI

**Task 1: OpenAI Setup**
1. Configure OpenAI API key using environment variables.
2. Initialize an OpenAI chat model using LangChain.


In [3]:
load_dotenv()

if os.getenv("OPENAI_API_KEY") is None:
    raise ValueError("OPENAI_API_KEY is not set")

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


**Task 2: Basic OpenAI Q&A Chatbot**
1. Create a ChatPromptTemplate with:
   - System message (role of assistant)
   - Human message (user question)
2. Pass user questions to the OpenAI model.
3. Print chatbot responses.

Test with at least 5 different questions.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

In [15]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can answer questions."),
    ("human", "{question}"),
])

chain_openai = prompt | llm

In [16]:
qsns = [
    "What is LLM?",
    "What is LangChain?",
    "How to use LangChain?",
    "What is the difference between OpenAI and Ollama?",
    "What is TPU and how it is used in training LLMs?"
]
for qsn in qsns:
    response = chain_openai.invoke({"question": qsn})
    print(f"Question: {qsn}")
    print(f"Response: {response.content}")
    print("-"*100)
    

Question: What is LLM?
Response: LLM stands for "Large Language Model." It refers to a type of artificial intelligence model that is designed to understand and generate human language. These models are trained on vast amounts of text data and use deep learning techniques, particularly neural networks, to learn patterns, grammar, facts, and even some reasoning abilities from the data.

LLMs can perform a variety of language-related tasks, including:

- Text generation
- Translation
- Summarization
- Question answering
- Sentiment analysis
- Conversational agents (chatbots)

Examples of large language models include OpenAI's GPT (Generative Pre-trained Transformer) series, Google's BERT (Bidirectional Encoder Representations from Transformers), and others. These models have become increasingly popular due to their ability to produce coherent and contextually relevant text, making them useful in many applications across different industries.
-----------------------------------------------

**Task 3: Multi-Turn Q&A (Optional)**
1. Maintain simple conversation history.
2. Ask follow-up questions.
3. Observe how OpenAI handles context.

**Observation**
- with `MessagesPlaceholder` + `RunnableWithMessageHistory`, follow-ups work (e.g. "What is my name?" after saying Abhishek)
- without history the model has no idea who "I" am — context only exists if we pass prior turns


In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the user's question based on the conversation history."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
])

In [12]:
store = {}
def get_chat_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain = prompt | llm | StrOutputParser()

chatbot = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [13]:
cfg = {'configurable': {'session_id': 'abhi1'}}
chatbot.invoke({'input': 'My name is Abhishek'}, config=cfg)
print(chatbot.invoke({'input': 'What is my name?'}, config=cfg))

Your name is Abhishek.


## PART 2 — Q&A Chatbot using Ollama (Open-Source Model)

**Task 4: Ollama Setup**
1. Install Ollama.
2. Pull an open-source model:
`ollama pull llama3`
3. Verify model is running locally.

In [ ]:
# Task 4 — verify ollama is up + model pulled
!ollama list



**Task 5: Ollama Chat Model with LangChain**
1. Initialize Ollama chat model using LangChain.
2. Use the same prompt template as OpenAI.
3. Generate responses from the Ollama model.

In [14]:
from langchain_ollama import ChatOllama

In [18]:
ollama = ChatOllama(model="llama3.2:3b", temperature=0.2)
chain_ollama = prompt | ollama

In [20]:
qsns = [
    "What is LLM?",
    "What is LangChain?",
    "How to use LangChain?",
]
for qsn in qsns:
    response = chain_ollama.invoke({"question": qsn})
    print(f"Question: {qsn}")
    print(f"Response: {response.content}")
    print("-"*100)

Question: What is LLM?
Response: LLM stands for Large Language Model. It's a type of artificial intelligence (AI) model designed to process and understand human language at a massive scale.

Large Language Models are trained on vast amounts of text data, which enables them to learn patterns, relationships, and context within language. This training allows the models to generate human-like responses, answer questions, summarize content, and even create new text based on a given prompt or topic.

Some key characteristics of Large Language Models include:

1. **Massive scale**: LLMs are trained on enormous amounts of data, often in the tens or hundreds of billions of parameters.
2. **Language understanding**: They can comprehend complex language structures, nuances, and context-dependent meanings.
3. **Generative capabilities**: LLMs can generate text that's coherent, natural-sounding, and often indistinguishable from human-written content.

Large Language Models have many applications, i

**Task 6: Compare OpenAI vs Ollama Outputs**
Answer briefly:

1. Response quality → OpenAI (`gpt-4o-mini`) usually clearer / more consistent on general Q&A. Ollama (`llama3.2:3b`) is decent for simple stuff but smaller local models can be shorter or a bit less polished.
2. Latency → OpenAI depends on network + API. Ollama is local so no API hop, but on CPU a 3B model can still feel slow; GPU helps a lot.
3. Cost → OpenAI = pay per token. Ollama = free after you own the machine (electricity / hardware only).
4. Privacy → Ollama keeps prompts on your machine. OpenAI sends data to their API (fine for many apps, not ideal for sensitive / offline use).


In [ ]:
# Task 6 answers
print("1. Quality: OpenAI usually stronger/cleaner; small Ollama models ok for basic Q&A.")
print("2. Latency: OpenAI = network; Ollama = local (fast on GPU, meh on CPU).")
print("3. Cost: OpenAI billed per token; Ollama free locally after hardware.")
print("4. Privacy: Ollama stays on-device; OpenAI sends prompts to the cloud.")



## PART 3 — Unified Q&A Chatbot App

**Task 7: Model Switch Logic**
1. Create a function:
```
def get_answer(question, model_type="openai"):
    # returns answer from selected model
```
2. Allow switching between:
   - OpenAI
   - Ollama


In [21]:
def get_answer(question, model_type="openai"):
    if model_type == "openai":
        response = chain_openai.invoke({"question": question})
    elif model_type == "ollama":
        response = chain_ollama.invoke({"question": question})
    else:
        raise ValueError(f"Invalid model type: {model_type}")
    return response.content

In [22]:
qsns = [
    "What is LLM?",
    "What is LangChain?",
    "How to use LangChain?",
]
for qsn in qsns:
    print(f"Question: {qsn}")
    print(f"OpenAI Response: {get_answer(qsn, 'openai')}")
    print(f"Ollama Response: {get_answer(qsn, 'ollama')}")
    print("-"*100)

Question: What is LLM?
OpenAI Response: LLM stands for "Large Language Model." It refers to a type of artificial intelligence model that is designed to understand and generate human language. These models are trained on vast amounts of text data and use deep learning techniques, particularly neural networks, to learn patterns, grammar, facts, and even some reasoning abilities from the data.

LLMs can perform a variety of language-related tasks, including:

- Text generation
- Translation
- Summarization
- Question answering
- Sentiment analysis
- Conversational agents (chatbots)

Examples of large language models include OpenAI's GPT-3 and GPT-4, Google's BERT, and others. These models have gained significant attention for their ability to produce coherent and contextually relevant text, making them useful in various applications across industries.
Ollama Response: LLM stands for Large Language Model. It's a type of artificial intelligence (AI) model designed to process and understand 

In [ ]:

basic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can answer questions."),
    ("human", "{question}"),
])
chain_openai = basic_prompt | llm
chain_ollama = basic_prompt | ollama


**Task 8: Build Simple App (CLI or Streamlit)**
Build a chatbot app that:
- Takes user questions
- Lets user choose model (OpenAI / Ollama)
- Displays answers clearly

**What I built**
- Streamlit app in `app.py`
- sidebar selectbox: `openai` / `ollama`
- text input + button → shows answer

Run:
```bash
cd assignment-29
streamlit run app.py
```


## PART 4 — Observations & Insights

**Task 9: Conceptual Questions**
Write short answers:

1. When to prefer OpenAI models → when you want strong quality fast, managed infra, and you’re ok paying + sending data to an API (prototypes, SaaS chat, most course demos).
2. When to prefer open-source models (Ollama) → offline / private data, no per-token bill, or you want full control of the model on your laptop/server.
3. Trade-offs in production → hosted APIs = easier ops + rate limits/cost/vendor lock-in. local/self-host = privacy + control, but you own GPUs, scaling, monitoring, model updates.
4. Cost and scalability → OpenAI scales by paying more (simple). Ollama/local scales by adding hardware + load balancing (harder). many teams start with API, move sensitive / high-volume workloads to self-host later.
